In [1]:
# import gurobipy as gp
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objs as go
import plotly.io as pio
import numpy as np
import pandas as pd

Loading Network

In [2]:
def load_edges_from_csv(path_edges: str):
    """
    CSV esperado: node_id1,node_id2,b,f_max
    b = costo base por arco
    f_max = capacidad base
    """
    edges = pd.read_csv(path_edges)
    edges = edges.rename(columns={
        "node_id1": "i",
        "node_id2": "j",
        "b": "c_base",
        "f_max": "U_base"
    })

    edges["e_id"] = np.arange(len(edges))
    return edges

def load_nodes_from_csv(path_nodes: str):
    """
    CSV esperado:
    node_id,d,p_min,p_max,c_var,is_generator,energy_type,instance
    d = demanda base (si no es generador)
    p_max = capacidad gen máxima (si es generador)
    """
    nodes = pd.read_csv(path_nodes)
    nodes = nodes.rename(columns={"node_id":"node"})
    return nodes

Defining the world $W$, the baseline for scenarios

In [8]:
def sample_one_scenario_W(nodes, edges, rng,
                          p_crisis=0.05,
                          # mediocristan
                          dem_sigma=0.10,
                          cost_sigma=0.05,
                          p_outage=0.02,
                          cap_drop=0.7,
                          # extremistan
                          crisis_dem_mult_mu=2.0,
                          crisis_dem_mult_sigma=0.25,
                          crisis_cost_mult=1.25,
                          correlated_outage_nodes=None,
                          correlated_outage_prob=0.8):

    # 1) Crisis o no
    crisis = rng.random() < p_crisis

    # 2) Demanda por nodo: D_iω
    D = nodes[["node"]].copy()
    D["D_base"] = nodes["d"].fillna(0.0)

    if not crisis:
        mult = rng.normal(1.0, dem_sigma, size=len(D))
        mult = np.clip(mult, 0.0, None)
    else:
        pareto_alpha = 2.0
        Y = rng.pareto(a=pareto_alpha, size=len(D))
        mult = 1.0 + Y

    D["D_w"] = D["D_base"] * mult
    D.loc[D["D_base"] == 0, "D_w"] = 0.0

    # 3) Costo por arco: c_eω
    c = edges[["e_id", "c_base"]].copy()

    if not crisis:
        cost_mult = rng.normal(1.0, cost_sigma, size=len(c))
        cost_mult = np.clip(cost_mult, 0.1, None)
    else:
        cost_mult = np.ones(len(c)) * crisis_cost_mult
    c["c_w"] = c["c_base"] * cost_mult

    # 4) Capacidad por arco: U_eω
    U = edges[["e_id", "U_base", "i", "j"]].copy()

    outage = rng.random(len(U)) < p_outage
    U_mult = np.where(outage, cap_drop, 1.0)

    if crisis and correlated_outage_nodes is not None:
        if rng.random() < correlated_outage_prob:
            affected = U["i"].isin(correlated_outage_nodes) | U["j"].isin(correlated_outage_nodes)
            U_mult = np.where(affected, 0.0, U_mult)

    U["U_w"] = U["U_base"] * U_mult

    # 5) Capacidad de generación por nodo: G_iω  ← NUEVO
    G = nodes[["node"]].copy()
    G["G_base"] = nodes["p_max"].fillna(0.0)

    if not crisis:
        gen_mult = rng.normal(1.0, 0.05, size=len(G))
        gen_mult = np.clip(gen_mult, 0.5, 1.0)
    else:
        gen_mult = rng.uniform(0.3, 1.0, size=len(G))

    G["G_w"] = G["G_base"] * gen_mult
    G.loc[G["G_base"] == 0, "G_w"] = 0.0  # nodos de demanda no generan

    return {
        "crisis": int(crisis),
        "D": D[["node", "D_w"]],
        "U": U[["e_id", "U_w"]],
        "c": c[["e_id", "c_w"]],
        "G": G[["node", "G_w"]],   # ← NUEVO
    }

Montecarlo simulation

In [9]:

def generate_massive_W(nodes, edges, n_samples=1000, seed=123,
                       correlated_outage_nodes=None):
    rng = np.random.default_rng(seed)
    scenarios = []
    for _ in range(n_samples):
        sc = sample_one_scenario_W(nodes, edges, rng,
                                   correlated_outage_nodes=correlated_outage_nodes)
        scenarios.append(sc)
    return scenarios


Clustering process

In [10]:
def scenario_features(sc, nodes, edges):
    """
    Convierte escenario a un vector feature para cluster:
    - demanda total
    - % capacidad perdida (promedio)
    - costo medio
    - indicador crisis
    """
    Dtot = sc["D"]["D_w"].sum()

    U_base = edges.set_index("e_id")["U_base"]
    U_eff  = sc["U"].set_index("e_id")["U_w"]
    cap_loss = 1.0 - (U_eff / U_base.replace(0, np.nan)).fillna(1.0)
    cap_loss_mean = cap_loss.mean()

    c_mean = sc["c"]["c_w"].mean()

    return np.array([Dtot, cap_loss_mean, c_mean, sc["crisis"]], dtype=float)

def reduce_to_K_scenarios(scenarios, nodes, edges, K=20, seed=123):
    """
    Reducción simple:
    - extrae features
    - aplica k-means
    - elige como representante el escenario más cercano al centroide (medoid)
    - prob de cada escenario = proporción de puntos asignados
    """
    from sklearn.cluster import KMeans

    X = np.vstack([scenario_features(sc, nodes, edges) for sc in scenarios])
    km = KMeans(n_clusters=K, random_state=seed, n_init=20)
    labels = km.fit_predict(X)
    centers = km.cluster_centers_

    reps = []
    probs = []
    #encontrar representante de cada cluster
    for k in range(K):
        idx = np.where(labels == k)[0]
        probs.append(len(idx) / len(scenarios))
        # elegir el más cercano al centroide
        dists = np.linalg.norm(X[idx] - centers[k], axis=1)
        rep_idx = idx[np.argmin(dists)]
        reps.append(scenarios[rep_idx])

    return reps, np.array(probs)

In [11]:
def export_Wprime(reps, probs, out_prefix="Wprime"):
    """
    Crea tres tablas:
    - demanda: (omega, node, D_iw)
    - capacidad: (omega, e_id, U_ew)
    - costos: (omega, e_id, c_ew)
    - prob: (omega, p_omega, crisis)
    """
    rows_D, rows_U, rows_c, rows_p, rows_G = [], [], [], [], []

    for w, (sc, p) in enumerate(zip(reps, probs), start=1):
        rows_p.append({"omega": w, "p_omega": float(p), "crisis": sc["crisis"]})

        for _, r in sc["D"].iterrows():
            rows_D.append({"omega": w, "node": int(r["node"]), "D_iw": float(r["D_w"])})
        for _, r in sc["U"].iterrows():
            rows_U.append({"omega": w, "e_id": int(r["e_id"]), "U_ew": float(r["U_w"])})
        for _, r in sc["c"].iterrows():
            rows_c.append({"omega": w, "e_id": int(r["e_id"]), "c_ew": float(r["c_w"])})
        for _, r in sc["G"].iterrows():
            rows_G.append({"omega": w, "node": int(r["node"]), "G_iw": float(r["G_w"])})

    dfD = pd.DataFrame(rows_D)
    dfU = pd.DataFrame(rows_U)
    dfc = pd.DataFrame(rows_c)
    dfp = pd.DataFrame(rows_p)
    dfG = pd.DataFrame(rows_G)

    dfD.to_csv(f"{out_prefix}_D.csv", index=False)
    dfU.to_csv(f"{out_prefix}_U.csv", index=False)
    dfc.to_csv(f"{out_prefix}_c.csv", index=False)
    dfp.to_csv(f"{out_prefix}_p.csv", index=False)
    dfG.to_csv(f"{out_prefix}_G.csv", index=False)

    return dfD, dfU, dfc, dfp, dfG


In [12]:
edges = load_edges_from_csv("edges.csv")
nodes = load_nodes_from_csv("nodes.csv")

instance = 19
nodes = nodes[nodes["instance"] == instance].reset_index(drop=True)
gens = nodes[nodes["is_generator"] == True]["node"].to_list()

print("Instancia elegida:", instance)
print("nodos totales:", len(nodes))
print("generadores:", gens)
print("Tipo de energía para la instancia", instance, ":")
print(nodes.groupby("energy_type")["node"].count())

# 1) W masivo
escenarios_W = generate_massive_W(nodes, edges, n_samples=1000, seed=1,
                                  correlated_outage_nodes=gens)

# 2) Reducir a 20 escenarios
reps, probs = reduce_to_K_scenarios(escenarios_W, nodes, edges, K=20, seed=1)

# 3) Exportar — ahora retorna 5 valores (dfG es nuevo)
dfD, dfU, dfc, dfp, dfG = export_Wprime(reps, probs, out_prefix="Wprime20")

Instancia elegida: 19
nodos totales: 118
generadores: [9, 11, 24, 25, 30, 45, 48, 53, 58, 60, 64, 65, 79, 86, 88, 99, 102, 110]
Tipo de energía para la instancia 19 :
energy_type
coal     4
gas      6
hydro    2
solar    4
wind     2
Name: node, dtype: int64


In [35]:
# nodes
# dfD[dfD["omega"]==1]
dfp["p_omega"].sum()

np.float64(1.0000000000000002)